In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# ---------------------------------------------------------------------
# Add project root to Python path
# ---------------------------------------------------------------------

# project root = two levels above notebooks
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
from src.data.spatial_clustering import create_spatial_clusters
from src.data.spatial_splitting import create_spatial_test_and_cv_splits

from config.paths import MAIN_DATASET
from config.paths import SPATIAL_CV_DATASET
from config.paths import SPATIAL_CV_DIR

# make sure those directories exist...
SPATIAL_CV_DIR.mkdir(parents=True, exist_ok=True)
# and that data is there...
if not MAIN_DATASET.exists():
    raise FileNotFoundError(
        f"Main dataset not found: {MAIN_DATASET}"
    )

# Ensure that we aren't overwriting old splits all the time!!!
if SPATIAL_CV_DATASET.exists():
    raise RuntimeError(
        f"Spatial CV split file already exists:\n{SPATIAL_CV_DATASET}\n"
        "Delete it manually if regeneration is intended."
    )

In [ ]:
# --------------------------------------------------
# Spatial CV configuration
# --------------------------------------------------

N_SPATIAL_CLUSTERS = 200  # number of k-means clusters that we use to spatially divide Canada

NUM_TEST_SPLITS = 5  # KFOLD SPLITTING NEEDS AN INTEGER: 1/N gives the fractional size of the test split
NUM_CV_SPLITS = 5  # Number of folds for kfold splitting
RANDOM_STATE = 42  # random seed


### Split up data between: 
  - Test data (label = '-1')
  - Cross validation data (label = 0 -- N-1, where N is number of folds )
  
  To use for CV, train on data with label (!=0 and != N_{current_CV_round})

In [ ]:
df = pd.read_csv(MAIN_DATASET)

gdf_clusters = create_spatial_clusters(df, n_clusters=N_SPATIAL_CLUSTERS)

df_splits = create_spatial_test_and_cv_splits(
    gdf_clusters,
    n_test_splits=NUM_TEST_SPLITS,
    n_cv_splits=NUM_CV_SPLITS,
    random_state=RANDOM_STATE
)

# Get rid of ID column from spatial-kfold work
df_splits = df_splits.drop(columns=["spkf_id"])

c:\Users\john\anaconda3\envs\erdos_ds_environment\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


### Write out data to CSV

In [ ]:
print("Saving dataset to:")
print(SPATIAL_CV_DATASET)
df_splits.to_csv(SPATIAL_CV_DATASET, index=False)

Saving dataset to:
C:\Users\john\Desktop\spring-2026-radon-risk-mapping\data\modeling\spatial_cv\dataset_with_spatial_cv_splits.csv


In [ ]:
cluster_summary = (
    df_splits
    .groupby("spatial_cluster")
    .agg(
        n_observations=("spatial_cluster", "size"),
        province=("provinceterritory", "first"),
        is_test=("is_test", "first"),
        cv_fold=("cv_fold", "first")
    )
    .reset_index()
)

In [ ]:
df_splits.head()

Index(['FSA', 'n_days', 'concentration', 'longitude', 'latitude',
       'hous_frac_type_single_detached', 'hous_frac_type_highrise',
       'hous_frac_type_other_attached', 'hous_frac_type_movable',
       'hous_avg_rooms', 'hous_frac_major_repair', 'hous_frac_age_pre_1980',
       'hous_frac_age_1981_2000', 'hous_frac_age_post_2001',
       'hous_median_value', 'demogr_pop_2016', 'demogr_num_total_dwellings',
       'demogr_num_occ_dwellings', 'demogr_median_age',
       'demogr_avg_household_size', 'demogr_frac_tenure_owned',
       'demogr_frac_tenure_rented', 'demogr_frac_tenure_band',
       'socioeco_frac_low_income', 'socioeco_median_income',
       'socioeco_frac_high_income', 'socioeco_frac_govt_transfers',
       'socioeco_frac_unemployment_rate', 'socioeco_frac_nonlaborer',
       'socioeco_frac_bachelor_plus', 'socioeco_frac_overcrowded',
       'socioeco_frac_unsuitable_housing', 'socioeco_frac_housing_burden',
       'rxtp_intrusive_rocks', 'rxtp_metamorphic_rocks',
    

,FSA,n_days,concentration,longitude,latitude,hous_frac_type_single_detached,hous_frac_type_highrise,hous_frac_type_other_attached,hous_frac_type_movable,hous_avg_rooms,...,sedi_weathered_bedrock_or_regolith,average_heating_days,mean_uranium,max_uranium,provinceterritory,geometry,spkf_id,spatial_cluster,is_test,cv_fold
0,V8J,122.0,7.5,-130.249226,54.47385,0.571573,0.007056,0.40121,0.020161,6.3,...,0.0,335.516156,NaN,NaN,BC,POINT (3901213.827 2747724.131),0,115,False,0
1,V8J,91.0,7.5,-130.249226,54.47385,0.571573,0.007056,0.40121,0.020161,6.3,...,0.0,335.516156,NaN,NaN,BC,POINT (3901213.827 2747724.131),1,115,False,0
2,V8J,91.0,7.5,-130.249226,54.47385,0.571573,0.007056,0.40121,0.020161,6.3,...,0.0,335.516156,NaN,NaN,BC,POINT (3901213.827 2747724.131),2,115,False,0
3,V8J,92.0,7.5,-130.249226,54.47385,0.571573,0.007056,0.40121,0.020161,6.3,...,0.0,335.516156,NaN,NaN,BC,POINT (3901213.827 2747724.131),3,115,False,0
4,V8J,93.0,7.5,-130.249226,54.47385,0.571573,0.007056,0.40121,0.020161,6.3,...,0.0,335.516156,NaN,NaN,BC,POINT (3901213.827 2747724.131),4,115,False,0


In [ ]:
# N_observations per fold
df_splits.groupby("cv_fold").size()

In [ ]:
#Largest cluster:
cluster_summary.sort_values("n_observations", ascending=False).head()

In [ ]:
# clusters per fold
cluster_summary.groupby("cv_fold").size()